In [1]:
import pymongo
client = pymongo.MongoClient("mongodb://localhost:27017/")
langweb_db = client['langweb']
verbs_collection = langweb_db['verbs']
games_collection = langweb_db['games']

In [2]:
from selenium import webdriver
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.chrome.options import Options
import unicodedata
options=Options()
driver = webdriver.Chrome()
verbs = []
def get_verbs(url):
    driver.get(url)
    WebDriverWait(driver, 1)
    driver.find_element("xpath" ,"/html/body/div[1]/div/div/div/div/div/div[2]/button[2]").send_keys(Keys.ENTER)
    frame_verb=driver.find_element("xpath" ,"//*[@id='form1']/div[3]/div/div[1]/div[4]/ol")
    
    elements = frame_verb.find_elements("tag name", "li")
    print(len(elements))
    verbs.extend([element.text for element in elements])
    WebDriverWait(driver, 1)
    driver.close()
interval="1-2000"
url = f"https://conjugator.reverso.net/index-german-{interval}.html"
get_verbs(url)
    


2000


In [97]:
def get_conjugations(verb):
    driver.find_element("xpath", "/html/body/div[2]/div[1]/div/div[1]/div/form/div[3]/div/div[1]/div[2]/div[1]/div/div[1]/input").send_keys(verb)
    driver.find_element("xpath", "/html/body/div[2]/div[1]/div/div[1]/div/form/div[3]/div/div[1]/div[2]/div[1]/div/a").send_keys(Keys.ENTER)
    options.page_load_strategy = 'eager'
    WebDriverWait(driver, 1)
    
driver = webdriver.Chrome()
driver.get("https://conjugator.reverso.net/conjugation-german.html")
WebDriverWait(driver, 1)
driver.find_element("xpath","/html/body/div[1]/div/div/div/div/div/span").click()
for verb in verbs[:5]:
    print(verb)

    get_conjugations(verb)

werden
einen
verfahren
haben
können


NoSuchWindowException: Message: no such window: target window already closed
from unknown error: web view not found
  (Session info: chrome=141.0.7390.123)
Stacktrace:
	GetHandleVerifier [0x0x7ff6c941e8e5+80021]
	GetHandleVerifier [0x0x7ff6c941e940+80112]
	(No symbol) [0x0x7ff6c91a060f]
	(No symbol) [0x0x7ff6c91782f1]
	(No symbol) [0x0x7ff6c92288be]
	(No symbol) [0x0x7ff6c9248fa2]
	(No symbol) [0x0x7ff6c9221003]
	(No symbol) [0x0x7ff6c91e95d1]
	(No symbol) [0x0x7ff6c91ea3f3]
	GetHandleVerifier [0x0x7ff6c96ddc7d+2960429]
	GetHandleVerifier [0x0x7ff6c96d7f3a+2936554]
	GetHandleVerifier [0x0x7ff6c96f8977+3070247]
	GetHandleVerifier [0x0x7ff6c94383ce+185214]
	GetHandleVerifier [0x0x7ff6c943fe1f+216527]
	GetHandleVerifier [0x0x7ff6c9427b24+117460]
	GetHandleVerifier [0x0x7ff6c9427cdf+117903]
	GetHandleVerifier [0x0x7ff6c940dbb8+11112]
	BaseThreadInitThunk [0x0x7ffbb0aee8d7+23]
	RtlUserThreadStart [0x0x7ffbb198c53c+44]


In [5]:
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
def create_dataset(verb):
    verb_obj = {}
    def openpage():
        WebDriverWait(driver, 5).until(EC.presence_of_element_located((By.XPATH, "/html/body/nav/div/div[1]/form/table/tbody/tr/td[1]/div/input[1]")))
        driver.find_element(By.XPATH, "/html/body/nav/div/div[1]/form/table/tbody/tr/td[1]/div/input[1]").send_keys(verb)
        driver.find_element(By.XPATH, "/html/body/nav/div/div[1]/form/table/tbody/tr/td[2]/button").click()
        WebDriverWait(driver, 5).until(EC.presence_of_element_located((By.XPATH,"//*[@id='vStckKrz']/div[2]/p[1]/span")))
    
    def verbinfo():
        meaning = driver.find_element(By.XPATH,"//*[@id='vStckKrz']/div[2]/p[1]/span").text
        otherinfo = driver.find_element(By.XPATH, "//*[@id='vVdBx']/p").text

        if otherinfo.split("·")[1].strip() == "unregelmäßig":
            regularity = False
        else: regularity = True

        root_change_info = driver.find_element(By.XPATH, "//*[@id='vStckKrz']/p[2]").text
        if root_change_info.strip():
            root_change = True
        else: root_change = False

        if otherinfo.split("·")[-1].strip() == "untrennbar":
            separability = False
        elif otherinfo.split("·")[-1].strip() == "trennbar": 
            separability = True
        else: separability = None
        info_dict = {"verb": verb,
                "meaning": meaning,
                "regular": regularity,
                "root_change": root_change,
                "separable": separability,
                "reflexive": False,
        }
        return info_dict
    
    def conjugations():
        parent = driver.find_element(By.CLASS_NAME, "rAbschnitt")
        indikativ_parent = parent.find_element(By.XPATH, "div/section[3]/div[2]")
        times = indikativ_parent.find_elements(By.CLASS_NAME, "vTbl")
        pronouns = ["ich","du","er/sie/es","wir","ihr", "sie/Sie"]
        indikativ_dict={}
        for time in times:
            tense= time.find_element(By.TAG_NAME,"h3").text
            tense = tense.replace(".","")
            indikativ_dict[tense]={}
            for pronoun in pronouns:
                conjugation = time.find_element(By.XPATH, f"table/tbody/tr[{pronouns.index(pronoun)+1}]/td[2]").text
                conjugation = ''.join(c for c in conjugation if unicodedata.category(c) != 'No' and c not in '()')
                indikativ_dict[tense][pronoun] = conjugation
        verb_obj["Indikativ"] = indikativ_dict
                

        konjunktiv_parent = parent.find_element(By.XPATH, "div/section[4]/div[2]")
        times = konjunktiv_parent.find_elements(By.CLASS_NAME, "vTbl")
        pronouns = ["ich","du","er/sie/es","wir","ihr", "sie/Sie"]
        konjunktiv_dict={}
        for time in times:
            tense= time.find_element(By.TAG_NAME,"h3").text
            tense = tense.replace(".","")
            konjunktiv_dict[tense]={}
            for pronoun in pronouns:
                conjugation = time.find_element(By.XPATH, f"table/tbody/tr[{pronouns.index(pronoun)+1}]/td[2]").text
                conjugation = ''.join(c for c in conjugation if unicodedata.category(c) != 'No' and c not in '()')
                konjunktiv_dict[tense][pronoun] = conjugation
        verb_obj["Konjunktiv"] = konjunktiv_dict
                
                
        imperativ_parent = parent.find_element(By.XPATH, "div/section[6]/div[2]")
        times = imperativ_parent.find_elements(By.CLASS_NAME, "vTbl")
        pronouns = ["du","wir","ihr", "Sie"]
        imperativ_dict={}
        for time in times:
            tense= time.find_element(By.TAG_NAME,"h3").text
            imperativ_dict[tense]={}
            for pronoun in pronouns:
                conjugation = time.find_element(By.XPATH, f"table/tbody/tr[{pronouns.index(pronoun)+1}]").text
                conjugation = ''.join(c for c in conjugation if unicodedata.category(c) != 'No' and c not in '()')
                imperativ_dict[tense][pronoun] = conjugation
        verb_obj["Imperativ"] = imperativ_dict


    openpage()
    info_dict = verbinfo()
    verb_obj= info_dict
    conjugations()
    verbs_collection.insert_one(verb_obj)
    print("data added")

    if verb_obj["separable"] != None:
        div = driver.find_element(By.XPATH, "/html/body/article/div[1]")
        children = div.find_elements(By.XPATH, "./a")
        number_of_children = len(children)
        for i in range(1, number_of_children):
            div = driver.find_element(By.XPATH, "/html/body/article/div[1]")
            child = div.find_elements(By.XPATH, "./a")[i]
            print(child)
            print(child.tag_name, child.get_attribute("outerHTML")[:150])

            href_value = child.get_attribute("href")
            img_value = child.find_element(By.TAG_NAME, "img").get_attribute("src")
            if href_value.startswith("https://www.verbformen.de/konjugation") and img_value == "https://www.verbformen.de/unselected.svg":
                driver.get(href_value)
                WebDriverWait(driver, 5).until(EC.presence_of_element_located((By.XPATH,"//*[@id='vStckKrz']/div[2]/p[1]/span")))
                info_dict = verbinfo()
                verb_obj= info_dict
                conjugations()
                WebDriverWait(driver, 10)
                verbs_collection.insert_one(verb_obj)
                print("data added")
            
    

    WebDriverWait(driver, 3)
    driver.find_element(By.XPATH, "/html/body/nav/div/div[1]/form/table/tbody/tr/td[1]/div/input[1]").clear()
    # print(verb_obj)
    return verb_obj
    
driver = webdriver.Chrome()
driver.get("https://www.verbformen.de/")
WebDriverWait(driver, 1)
WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.XPATH,"/html/body/div[6]/iframe")))
frame_cookies= driver.find_element(By.XPATH, "/html/body/div[6]/iframe")
driver.switch_to.frame(frame_cookies)
WebDriverWait(driver, 10).until(EC.element_to_be_clickable((By.XPATH,"/html/body/div/div[2]/div[5]/button"))).click()
driver.switch_to.default_content()

WebDriverWait(driver, 1)
options.page_load_strategy = 'eager'
# for verb in verbs[:1]:
#     print(verb)
#     verb_obj = create_dataset(verb)
    # 
verb_obj = create_dataset("unternehmen")
verb_obj = create_dataset("umziehen")
verb_obj = create_dataset("werden")
verb_obj = create_dataset("einen")
driver.close()

data added
<selenium.webdriver.remote.webelement.WebElement (session="c32fad4439b567a8ad3885f9793b56a5", element="f.48C30FE2F28F6B2DDB59D2C48C23C954.d.6549D72D16AAB9C79DCF75DA61AAA739.e.229")>
a <a class="rKnpf rNoSelect rLinks" href="/konjugation/unter-nehmen.htm" title="unter-nehmen">
<div class="rKln rInf" style="margin-bottom: 5px;">
<span
data added
<selenium.webdriver.remote.webelement.WebElement (session="c32fad4439b567a8ad3885f9793b56a5", element="f.48C30FE2F28F6B2DDB59D2C48C23C954.d.36C7F93E5A7EC659B5995114DD8BC7EF.e.373")>
a <a class="rKnpf rNoSelect rLinks" href="/deklination/substantive/Unternehmen.htm" title="Unternehmen">
<div class="rKln rInf" style="margin-bottom: 5p
data added
<selenium.webdriver.remote.webelement.WebElement (session="c32fad4439b567a8ad3885f9793b56a5", element="f.48C30FE2F28F6B2DDB59D2C48C23C954.d.49CF73ACC8ABD30B6415D19591C82B09.e.513")>
a <a class="rKnpf rNoSelect rLinks" href="/konjugation/um-ziehen_hat.htm" title="um-ziehen (hat)">
<div class="rKln

Verb
Meaning
Regularity
Separability
Reflexivity
Tense1
    Pronoun1
    Pronoun2
    ...
Tense2
    Pronoun1
    Pronoun2
    ... 

